# B2-Li + DG + BiFPN перед EMCAD
EMCAD сохранён. Cross-attention отключён. 6 эпох, последние 2 на полных кадрах.
Выберите control или BiFPN ширины 64/128 с 1/2 повторениями. По умолчанию все сравниваются на 704.
use_max_size=True включает измеренное максимальное разрешение для каждого варианта и отдельное имя run; это меняет два фактора сравнения.
Бюджет проверяется при native JPEG Full HD. Latency на H100 не измерена.
Перед запуском проверьте серверные пути, GPU, batch и accumulation в .env. Последняя ячейка запускает выбранный эксперимент.


In [ ]:
import numpy as np  # Import before torch on Windows (MKL initialization).
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open this notebook from the project root or notebooks directory')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)

from src.config import load_experiment_config

arm = 'w64_r2'  # control, w64_r1, w64_r2, w128_r1, w128_r2
use_max_size = False
choices = {'control': 'control', 'w64_r1': 'w64_r1', 'w64_r2': 'w64_r2', 'w128_r1': 'w128_r1', 'w128_r2': 'w128_r2'}
experiment = 'experiments/disentangle_b2_li704_bifpn_' + choices[arm]
cfg = load_experiment_config(ROOT / 'configs' / f'{experiment}.yaml')
if use_max_size:
    from dataclasses import replace
    sizes = {'control': 768, 'w64_r1': 752, 'w64_r2': 752, 'w128_r1': 744, 'w128_r2': 736}
    size = sizes[arm]
    cfg = replace(cfg, run_name=cfg.run_name.replace('li704', f'li{size}'),
                  dataset=replace(cfg.dataset, image_size=size))
print('Run:', cfg.run_name)
print('Encoder / RGB:', cfg.model.encoder, cfg.dataset.image_size)
print('Epochs / full passes:', cfg.train.epochs, cfg.train.full_pass_epochs)
print('Devices / batch per GPU / accumulation:', cfg.train.devices, cfg.train.batch_size, cfg.train.grad_accum_steps)
print('Data:', cfg.paths.data_path)
print('Runs:', cfg.paths.runs_path)
cfg


In [ ]:
import torch
from src.training.builders import build_model
from src.budget import count_gflops
from src.eval.protocol import EvaluationProtocol

protocol = EvaluationProtocol.load(cfg.dataset.protocol_path)
print('Train/development:', len(protocol.rows('train')), len(protocol.rows('development')))
native_size = (1080, 1920)
with torch.device('meta'):
    budget_model = build_model(cfg.model, aux_weight=cfg.loss.aux_weight, pretrained=False).eval()
    gflops = count_gflops(budget_model, cfg.dataset.image_size,
                          native_size=native_size)
del budget_model
assert gflops <= 100, f'{gflops:.2f} GFLOPs exceeds 100'
print(f'Full inference: {gflops:.3f} GFLOPs')
if native_size is not None:
    print('Reference native JPEG size:', native_size, '; larger sources may exceed 100 GFLOPs')


In [ ]:
from src.training.engine import run_experiment

run = run_experiment(cfg)
run.summary
